# 📓 Notebook 7 — NumPy Fundamentals: Vectorised Math for AI Workloads

> **Module:** Data Science Libraries · **Estimated time:** 35–50 min · **Difficulty:** Beginner / Intermediate

NumPy ("Numerical Python") is the bedrock of the entire Python data-science ecosystem. **Pandas, scikit-learn, TensorFlow, PyTorch, OpenCV** — all of them store data as NumPy arrays under the hood. Mastering NumPy is the single biggest leverage move in your early data-science journey.

The reason to care, for AI work specifically, is that **everything** — token counts, costs, latencies, probabilities, embeddings, model weights — eventually lives in a NumPy array. The same vectorised arithmetic you'll learn here is what makes `model.predict(X)` work over millions of rows in seconds.

## 🎯 Learning objectives

1. Create and inspect NumPy arrays (`shape`, `dtype`, `ndim`).
2. Index and slice 1-D and 2-D arrays, including with **boolean masks**.
3. Perform **vectorised arithmetic** and use **ufuncs**.
4. Understand **broadcasting** — how arrays of different shapes combine.
5. Aggregate **along an axis** (`axis=0` vs `axis=1`).
6. Reshape, stack, and split arrays.
7. Generate **reproducible** random data with `default_rng`.
8. Apply all of the above to a small AI-cost-and-latency analysis.

## ✅ Prerequisites

Notebooks 1–6.

## 1. Why NumPy?

For small datasets, pure Python is fine. For any serious numerical work, three problems with Python lists become unbearable:

1. **Slow** — Python loops are interpreted; NumPy runs vectorised C code under the hood (often 50–500× faster).
2. **Memory-hungry** — every `int` in a Python list is a full object; a NumPy array of 1 million `int32` values uses ~4 MB instead of ~28 MB.
3. **Awkward syntax** — element-wise arithmetic on lists requires a loop or comprehension; in NumPy it's just `a + b`.

In [ ]:
import numpy as np
import math
import time

# Pretend we have 5 million numeric records to process — common scale in analytics
n = 5_000_000
py_list = list(range(n))
np_arr  = np.arange(n)

# A small bit of real work per element: sqrt(x*x + x), then sum
# (e.g., a per-record score that's not just a single multiply)
t0 = time.perf_counter()
s1 = sum(math.sqrt(x*x + x) for x in py_list)
t_py = time.perf_counter() - t0

# Same formula, NumPy-vectorised
t0 = time.perf_counter()
s2 = np.sqrt(np_arr * np_arr + np_arr).sum()
t_np = time.perf_counter() - t0

assert abs(s1 - s2) / max(s1, 1) < 1e-6
print(f"Pure Python : {t_py*1000:7.1f} ms")
print(f"NumPy       : {t_np*1000:7.1f} ms   (≈ {t_py/t_np:.0f}× faster)")


## 2. Creating arrays

A handful of constructors you will use constantly:

| Call                          | Result                                    |
|-------------------------------|-------------------------------------------|
| `np.array([1, 2, 3])`         | from a Python list                        |
| `np.zeros((3, 4))`            | shape `(3, 4)`, all zeros                 |
| `np.ones((2, 5))`             | shape `(2, 5)`, all ones                  |
| `np.full((2, 2), 7)`          | filled with a constant                    |
| `np.arange(0, 10, 2)`         | 0, 2, 4, 6, 8                              |
| `np.linspace(0, 1, 5)`        | 5 evenly spaced values between 0 and 1     |
| `np.eye(3)`                   | 3×3 identity matrix                       |

In [ ]:
a = np.array([1, 2, 3, 4, 5])             # from a list
b = np.zeros((3, 4))                       # all zeros
c = np.ones((2, 5))                        # all ones
d = np.full((2, 2), 7)                     # filled
e = np.arange(0, 10, 2)                    # like range, but an array
f = np.linspace(0, 1, 5)                   # 5 points between 0 and 1
g = np.eye(3)                              # identity matrix

for name, arr in [("a", a), ("b", b), ("c", c), ("d", d), ("e", e), ("f", f), ("g", g)]:
    print(f"{name}: shape={arr.shape}, dtype={arr.dtype}")
    print(arr)
    print()


## 3. Inspecting an array

The four attributes you check first:

- `shape` — a tuple `(rows, cols, ...)`.
- `dtype` — element data type (`int64`, `float64`, `bool`, …).
- `ndim` — number of dimensions.
- `size` — total number of elements.

In [ ]:
# 3 models × 4 daily latency stats (mean, p50, p95, p99) in ms
latency_stats = np.array([
    [1820, 1740, 2950, 3820],     # gpt-4o-mini
    [2410, 2280, 3640, 4480],     # claude-haiku
    [1530, 1450, 2510, 3170],     # internal-llm
])

print(f"shape : {latency_stats.shape}")    # (3, 4)
print(f"dtype : {latency_stats.dtype}")
print(f"ndim  : {latency_stats.ndim}")
print(f"size  : {latency_stats.size}")
print(f"max   : {latency_stats.max()}")
print(f"min   : {latency_stats.min()}")
print(f"mean  : {latency_stats.mean():.1f}")


> 💡 **In ML and AI, `shape` is everything.** Most cryptic errors ("expected (32, 4) got (4, 32)") are shape mismatches. When you debug numerical code, *always* `print(x.shape)` first.

## 4. Indexing and slicing

For 1-D arrays this is identical to Python lists. For 2-D, the syntax is `A[row, col]` with slicing on each axis.

In [ ]:
# Index by (row, col) — same matrix from the previous cell
print("Model 0 (all stats)        :", latency_stats[0])           # whole row
print("Model 0 mean latency       :", latency_stats[0, 0])         # one cell
print("P95 column across models    :", latency_stats[:, 2])        # whole column
print("Bottom-right 2×2 sub-matrix :")
print(latency_stats[-2:, -2:])


### Boolean masking — the most powerful indexing pattern

A boolean array of the same shape can be used to index — only the `True` positions are returned. This is how you do "give me all calls where latency > 2 seconds" in pure NumPy and (with extra wrapping) in pandas.

In [ ]:
# A batch of 8 individual call latencies (ms)
latencies = np.array([1820, 2150, 1640, 3180, 2410, 1740, 2530, 1380])
print(f"latencies     = {latencies}")

mask = latencies > 2000
print(f"\nlatencies > 2000  → mask = {mask}")
print(f"latencies[mask]        = {latencies[mask]}")

# Idiomatic one-liner
print(f"latencies[latencies > 2000] = {latencies[latencies > 2000]}")

# Combine masks with & (AND) and | (OR) — parentheses required!
in_band = (latencies >= 1500) & (latencies <= 2500)
print(f"\n1.5–2.5s band : {latencies[in_band]}")


## 5. Vectorised arithmetic

This is what makes NumPy *feel* like maths: write the formula once, NumPy applies it to every element. No loops.

In [ ]:
# Convert latencies from ms to seconds — one expression, no loop
latencies_ms = np.array([1820, 2150, 1640, 3180, 2410, 1740, 2530, 1380])
latencies_s  = latencies_ms / 1000
print(f"ms : {latencies_ms}")
print(f"s  : {latencies_s}")

# Element-wise arithmetic between arrays of the same shape
tokens_in  = np.array([480, 720, 305, 610, 815, 290, 720, 410])
tokens_out = np.array([120, 200,  80, 175, 240,  95, 215, 130])

# Cost of every call in one vectorised expression — no Python loop required
price_in_per_1k  = 0.0006
price_out_per_1k = 0.0024

cost = tokens_in / 1000 * price_in_per_1k + tokens_out / 1000 * price_out_per_1k
print(f"\ncost (USD) per call:")
print(cost.round(5))
print(f"total cost: ${cost.sum():.5f}")


### Universal functions (ufuncs)

NumPy provides element-wise versions of every common math function. They are *fast* and they handle entire arrays at once. You'll see `np.exp`, `np.log`, `np.sqrt`, `np.abs`, `np.sin`, `np.cos`, … everywhere in ML code (especially in loss functions, softmax, attention).

In [ ]:
x = np.linspace(0, 2 * np.pi, 5)
print(f"x       : {x.round(3)}")
print(f"sin(x)  : {np.sin(x).round(3)}")
print(f"cos(x)  : {np.cos(x).round(3)}")
print(f"exp(x)  : {np.exp(x).round(3)}")
print(f"log1p(x): {np.log1p(x).round(3)}")
print(f"sqrt(x) : {np.sqrt(x).round(3)}")


## 6. Broadcasting — the rule that powers most of NumPy

What if two arrays have *different* shapes? NumPy tries to *stretch* the smaller one to match. The rule is:

> Compare shapes from the **right**. Dimensions are compatible when they are **equal** or one of them is **1**.

Two examples make this concrete.

In [ ]:
# Example 1: array + scalar — scalar broadcasts to every element
a = np.array([1, 2, 3, 4])
print(f"a + 100 = {a + 100}")

# Example 2: row vector + matrix
# Use case: we have a matrix of token counts for many calls × 2 columns (in, out)
# and we want to multiply each column by its own per-1K price.
calls = np.array([
    [480, 120],
    [720, 200],
    [305,  80],
    [610, 175],
])
prices_per_1k = np.array([0.0006, 0.0024])   # shape (2,)

# Cost = element-wise multiply then sum across the (in, out) axis
per_call_cost = (calls / 1000 * prices_per_1k).sum(axis=1)
print(f"\ncosts: {per_call_cost.round(5)}")
print(f"total: ${per_call_cost.sum():.5f}")


> 🎯 **Mental model.** "Compatible shapes" means: line them up from the right; each pair of dimensions must be equal, or one must be 1. Then the array with the 1 gets stretched to match. This one rule replaces what would otherwise be hundreds of nested loops.

## 7. Aggregations along an axis

Sum, mean, max, min — these all work over the whole array by default, but you can ask for them along a specific **axis**.

- `axis=0` → collapse rows → one value per column.
- `axis=1` → collapse columns → one value per row.

```
                     model 0   model 1   model 2
   mean latency      1820      2410      1530
   p50 latency       1740      2280      1450     ← sum(axis=1) → one value per row (stat)
   p95 latency       2950      3640      2510
   p99 latency       3820      4480      3170

   sum(axis=0)       ↓        ↓        ↓         (one value per column = per model)
```

In [ ]:
# Same matrix as section 3: 3 models × 4 latency stats
latency_stats = np.array([
    [1820, 1740, 2950, 3820],     # gpt-4o-mini
    [2410, 2280, 3640, 4480],     # claude-haiku
    [1530, 1450, 2510, 3170],     # internal-llm
])

print(f"sum()            : {latency_stats.sum()}              (everything)")
print(f"mean(axis=0)     : {latency_stats.mean(axis=0)}   (mean of each stat across models)")
print(f"mean(axis=1)     : {latency_stats.mean(axis=1)}   (mean of each model across stats)")
print()
print(f"argmin(axis=0)   : {latency_stats.argmin(axis=0)}        (fastest model for each stat)")
print(f"argmax(axis=1)   : {latency_stats.argmax(axis=1)}        (worst stat for each model)")


## 8. Reshape, stack, split

Manipulating shapes is one of the most common operations in ML — you'll reshape feature matrices, stack batches, split features and labels.

In [ ]:
x = np.arange(12)
print(f"x : shape={x.shape} → {x}")

# reshape — must keep total size the same
m34 = x.reshape(3, 4)
m43 = x.reshape(4, 3)
print(f"\nreshape (3,4):\n{m34}")

# Use -1 to let NumPy figure out one dimension automatically
auto = x.reshape(-1, 4)
print(f"\nreshape (-1, 4): shape={auto.shape}\n{auto}")

# Flatten back to 1-D
print(f"\nflatten : {m34.flatten()}")


In [ ]:
# Stacking arrays
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

print("hstack (1-D concat) :", np.hstack([a, b]))
print("vstack (stack rows) :\n", np.vstack([a, b]))
print("column_stack (cols) :\n", np.column_stack([a, b]))


## 9. Random data — reproducibly

For experiments and demos you want random data that *isn't actually random*: you want the same numbers every time you run the notebook so results are reproducible. Use `np.random.default_rng(seed)`.

In [ ]:
rng = np.random.default_rng(seed=42)

print("rng.random((3, 4)):")
print(rng.random((3, 4)).round(3))

print("\nrng.integers(low=0, high=10, size=(2, 5)):")
print(rng.integers(0, 10, size=(2, 5)))

print("\nrng.normal(mean=50, std=15, size=8):")
print(rng.normal(50, 15, size=8).round(2))

print("\nrng.choice(['gpt-4o-mini','claude-haiku'], size=5):")
print(rng.choice(["gpt-4o-mini", "claude-haiku"], size=5))


> 💡 The old `np.random.seed` / `np.random.rand` API still works, but the **Generator API** (`default_rng`) is the modern way: explicit, isolated, and it supports parallel streams when you need them.

## 10. Putting it all together — an A/B latency analysis

Suppose your team is comparing **two LLM providers** on the same 50 prompts. We will simulate the latencies, then ask: which provider is faster *on average*, by how much, and is the difference statistically meaningful?

This combines indexing, vectorised arithmetic, axis aggregation, and `argsort`.

In [ ]:
rng = np.random.default_rng(seed=0)

n = 50
# Simulate: provider A has mean 2.0s with std 0.5s; provider B has 2.4s with std 0.7s
latency_a = rng.normal(loc=2.0, scale=0.5, size=n).clip(min=0.2)
latency_b = rng.normal(loc=2.4, scale=0.7, size=n).clip(min=0.2)

# Stack into a (50, 2) matrix so we can use axis ops
both = np.column_stack([latency_a, latency_b])
labels = np.array(["A", "B"])

print(f"shape         : {both.shape}")
print(f"mean (axis=0) : {both.mean(axis=0).round(3)}  (one per provider)")
print(f"median        : {np.median(both, axis=0).round(3)}")
print(f"std           : {both.std(axis=0).round(3)}")
print(f"max           : {both.max(axis=0).round(3)}")

# Which prompts (rows) did A beat B on?
a_won = latency_a < latency_b
print(f"\nA was faster on {a_won.sum()} / {n} prompts ({a_won.mean():.0%})")

# A simple "effect size": mean difference / pooled std
diff = latency_b - latency_a
print(f"Mean difference (B - A): {diff.mean():.3f} s")
print(f"Effect size            : {diff.mean() / diff.std():.2f}  (≈ Cohen's d, rough rule: >0.5 is meaningful)")


**What just happened?**

- We simulated two parallel batches of 50 latencies using **`default_rng`**.
- We stacked them with **`np.column_stack`** so each *row* is a paired observation.
- **`axis=0`** gave us per-provider stats (mean, median, std).
- A **boolean mask** (`latency_a < latency_b`) counted how often A beat B.
- We computed a simple **effect size** without any extra library.

That's the entire shape of an A/B-test analysis in seven lines of code.

## 🧪 Practice exercises

### Exercise 1 — Create and slice

1. Create a 4×4 array containing the numbers 1–16.
2. Print the second row.
3. Print the third column.
4. Print the **bottom-right 2×2** sub-matrix.

In [ ]:
# Your code here  👇
import numpy as np


<details>
<summary>💡 <b>Solution</b></summary>

```python
A = np.arange(1, 17).reshape(4, 4)
print(A)
print()
print("row 1 :", A[1])
print("col 2 :", A[:, 2])
print("bottom-right 2×2:\n", A[-2:, -2:])
```
</details>

### Exercise 2 — Vectorised z-score

Given a 1-D array `x` of customer satisfaction scores, compute its **z-score**: subtract the mean, divide by the standard deviation. Verify that the result has mean ≈ 0 and std ≈ 1.

In [ ]:
# Your code here  👇
x = np.array([3.5, 4.1, 4.3, 2.9, 3.8, 4.7, 3.2, 4.0, 4.5, 3.6])


<details>
<summary>💡 <b>Solution</b></summary>

```python
z = (x - x.mean()) / x.std()
print("z         :", z.round(3))
print("z.mean()  :", round(float(z.mean()), 6))
print("z.std()   :", round(float(z.std()),  6))
```

Standardising features so they have mean 0 and std 1 is one of the most common preprocessing steps in ML — you'll see scikit-learn's `StandardScaler` do exactly this in NB 15.
</details>

### Exercise 3 — Boolean filtering on latencies

Given an array of call latencies in milliseconds, print:

1. The number of *slow* calls (latency > 3000 ms).
2. The mean latency of *fast* calls (< 1500 ms).
3. A new array where any latency above 5000 is clipped to 5000.

In [ ]:
# Your code here  👇
latencies = np.array([1850, 2510, 3520, 1240, 2810, 4220, 980, 1990, 3110, 870, 2240, 4870])


<details>
<summary>💡 <b>Solution</b></summary>

```python
n_slow = (latencies > 3000).sum()
print(f"Slow calls         : {n_slow}")

fast_mean = latencies[latencies < 1500].mean()
print(f"Mean of fast calls : {fast_mean:.1f} ms")

clipped = np.clip(latencies, None, 5000)
print(f"Clipped            : {clipped}")
```

**Pattern.** `(condition).sum()` counts how many entries satisfy a condition — `True` and `False` are summed as `1` and `0`. You'll use this trick constantly when evaluating models ("how many predictions were correct?").
</details>

### Exercise 4 — Broadcasting in action

You have a 10×3 feature matrix `X` where columns are (`age`, `monthly_spend`, `nps_score`). Standardise **each column** to have mean 0 and std 1 — i.e. subtract the column mean and divide by the column std. Do it in **one expression** using broadcasting (no loops).

In [ ]:
# Your code here  👇
rng = np.random.default_rng(0)
X = rng.normal(loc=[35, 80, 7], scale=[10, 25, 1.5], size=(10, 3))
print("Original X (rounded):\n", X.round(2))
print("\nCol means :", X.mean(axis=0).round(3))
print("Col stds  :", X.std(axis=0).round(3))


<details>
<summary>💡 <b>Solution</b></summary>

```python
X_std = (X - X.mean(axis=0)) / X.std(axis=0)
print("Standardised X:\n", X_std.round(3))
print("\nCheck means :", X_std.mean(axis=0).round(6))
print("Check stds  :", X_std.std(axis=0).round(6))
```

`X.mean(axis=0)` has shape `(3,)`, and broadcasting stretches it across all 10 rows — same for the std. The result should have mean ≈ 0 and std ≈ 1 per column.
</details>

### Exercise 5 — Debug me 🐞

The cell below should compute the **column means** of `A` but the output looks suspicious. Find and fix the bug.

In [ ]:
# 👇 Your fixed/corrected version goes here — write or paste it below.
A = np.arange(1, 13).reshape(3, 4)
print("A:\n", A)

means = A.mean(axis=1)         # bug
print("\nColumn means:", means)


<details>
<summary>💡 <b>Solution</b></summary>

`axis=1` collapses **columns** to produce one value per **row** — i.e. row means, not column means. For column means use `axis=0`.

```python
means = A.mean(axis=0)
print("Column means:", means)
```

The mnemonic: **`axis=k` means "this axis disappears"**. Row index is axis 0, column index is axis 1.
</details>

## 🧠 Stretch exercises

Two more applied exercises to deepen the material. Try them yourself before opening the solution.


### Stretch exercise A — Detect outliers (>3σ from the mean)

Write `find_outliers(arr, k=3)` that returns the *indices* of values more than `k` standard deviations from the mean. Test on `np.array([1, 2, 3, 4, 5, 100])`.


<details>
<summary>💡 <b>Solution</b></summary>

```python
def find_outliers(arr, k=3):
    z = (arr - arr.mean()) / arr.std(ddof=1)
    return np.where(np.abs(z) > k)[0]

test = np.array([1, 2, 3, 4, 5, 100])
idx  = find_outliers(test, k=2)        # k=2 to catch the obvious outlier
print(f"Outlier indices: {idx.tolist()}")
print(f"Outlier values : {test[idx].tolist()}")
```

**The "z-score" detector** is the textbook starting point for
outlier detection. It assumes roughly-normal data; for skewed
distributions you'd use the IQR rule (`Q1 − 1.5·IQR` / `Q3 + 1.5·IQR`)
— which is what `boxplot` draws its whiskers from.

</details>

### Stretch exercise B — Law of large numbers — running mean

Simulate 5,000 coin flips (Bernoulli p=0.5) with NumPy. Compute the **running proportion of heads** at every step, and verify the proportion at step 5,000 is within 1% of 0.5.


<details>
<summary>💡 <b>Solution</b></summary>

```python
rng = np.random.default_rng(42)
flips = rng.integers(0, 2, size=5_000)
running = np.cumsum(flips) / np.arange(1, len(flips) + 1)

print(f"After 100   flips: {running[99]:.3f}")
print(f"After 1,000 flips: {running[999]:.3f}")
print(f"After 5,000 flips: {running[-1]:.3f}")
assert abs(running[-1] - 0.5) < 0.01, "off by more than 1%"
print("\n✓ within 1% of 0.5 — Law of Large Numbers holds.")
```

**`cumsum / arange`** is the vectorised one-liner for a running mean.
Same pattern works for running totals, running averages of a
metric, exponential-weighted moving averages (with `.ewm()` in pandas).

</details>

### Stretch exercise C — Vectorised rolling mean

Compute the centred 5-element rolling mean of `x = np.arange(1, 21)` *without* using pandas or a Python loop. Use `np.convolve` with a uniform kernel.

Expected output: 16 values (the 4 edge positions are dropped); the first one should be 3.0.

In [ ]:
# Your code here  👇
import numpy as np
x = np.arange(1, 21)

# ...


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
import numpy as np
x = np.arange(1, 21)

k = 5
kernel = np.ones(k) / k
rolling = np.convolve(x, kernel, mode="valid")
print(rolling)         # length 16, first value 3.0
```

**Reasoning.** A rolling mean is convolution with a uniform kernel — that's the same operation image processing uses for blur. The three `mode=` options matter. `"valid"` returns only positions where the kernel fully overlaps the data (cleanest result, shortest output); `"same"` pads the output to the same length as `x` (convenient but the edges are biased); `"full"` returns every position the kernel touches the data (longest, edge-heavy). For rolling means in analysis, you almost always want `"valid"`. The same trick works for moving median (with a different kernel — see `scipy.signal.medfilt`).
</details>

### Stretch exercise D — Pairwise distance via broadcasting

Compute the pairwise Euclidean distances between every row of an array of 5 random 2-D points. Use **broadcasting**, not Python loops. The result should be a 5 × 5 symmetric matrix with zeros on the diagonal.

```python
pts = np.array([[0,0],[1,0],[0,1],[1,1],[2,2]], dtype=float)
```

In [ ]:
# Your code here  👇
import numpy as np
pts = np.array([[0,0],[1,0],[0,1],[1,1],[2,2]], dtype=float)

# ...


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
import numpy as np
pts = np.array([[0,0],[1,0],[0,1],[1,1],[2,2]], dtype=float)

# pts is (n, d). Reshape to (n, 1, d) and (1, n, d), subtract via broadcasting.
diff = pts[:, None, :] - pts[None, :, :]      # shape (n, n, d)
D = np.sqrt((diff ** 2).sum(axis=-1))         # shape (n, n)
print(np.round(D, 3))
```

**Reasoning.** This is the classic broadcasting trick: reshape one input to `(n, 1, d)` and the other to `(1, n, d)` and subtract; NumPy expands both to `(n, n, d)` without making any copies. Summing the squared difference along the **last axis** (`axis=-1`) collapses the `d` dimension, leaving an `n × n` matrix. Two follow-ups worth knowing. (1) For very large `n` you should use `scipy.spatial.distance.cdist(pts, pts)` — it's implemented in C and uses less memory. (2) The exact same pattern computes cosine distances if you L2-normalise the rows first and replace the subtraction with a dot product.
</details>

## 🎁 Bonus mini-project — Simulate an A/B test of two models

Use `rng.normal(loc, scale, size=n)` to simulate `n` satisfaction scores for two candidate AI models. Try `n = 30`, `n = 200`, `n = 5000`:

1. Compute mean and std of each model's scores.
2. Compute the **difference of means** and how many percentage points B beats A by.
3. Bonus: plot the **running mean** of each model's score vs sample number, to see how each estimate stabilises (the **law of large numbers** in action).

In [ ]:
# Your code here  👇
rng = np.random.default_rng(0)
# Suggested means/stds: A ~ N(4.0, 0.6), B ~ N(4.2, 0.6)


<details>
<summary>💡 <b>Solution</b></summary>

```python
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

for n in [30, 200, 5_000]:
    a = rng.normal(4.0, 0.6, size=n)
    b = rng.normal(4.2, 0.6, size=n)
    print(f"n={n:>5} : mean A = {a.mean():.3f}, mean B = {b.mean():.3f}, "
          f"diff = {b.mean() - a.mean():+.3f}")

# Running-mean plot
n = 3_000
a = rng.normal(4.0, 0.6, size=n)
b = rng.normal(4.2, 0.6, size=n)
running_a = np.cumsum(a) / np.arange(1, n + 1)
running_b = np.cumsum(b) / np.arange(1, n + 1)

plt.figure(figsize=(9, 4))
plt.axhline(4.0, color="grey", ls="--", alpha=0.5, label="true A=4.0")
plt.axhline(4.2, color="black", ls="--", alpha=0.5, label="true B=4.2")
plt.plot(running_a, label="Model A running mean")
plt.plot(running_b, label="Model B running mean")
plt.xlabel("Sample number")
plt.ylabel("Running mean satisfaction")
plt.title("Law of large numbers — running mean of two models")
plt.legend()
plt.tight_layout()
plt.show()
```

**What you should see.** At n=30 the noise can flip the result either way. By n=200 B is reliably ahead. By n=5000 both running means have nearly converged to their true values. This is exactly why production A/B tests need *sample-size discipline* before drawing conclusions.
</details>

## 🧠 Key takeaways

1. NumPy arrays are **homogeneous**, **typed**, and **vectorised** — that's why they're fast.
2. Always know your array's `shape` and `dtype`; most ML bugs are shape mismatches.
3. **Indexing / slicing** works on each axis; **boolean masks** let you filter.
4. **Vectorised arithmetic** + **ufuncs** replace explicit loops — write the formula, not the loop.
5. **Broadcasting** stretches arrays of compatible shapes — one rule replaces dozens of loops.
6. **`axis=0`** = collapse rows (per-column result); **`axis=1`** = collapse columns (per-row result).
7. Reproducible randomness: `rng = np.random.default_rng(seed=...)`.
8. The same patterns scale: from 8 numbers in a notebook to millions of rows in a production pipeline.

## ✅ Self-assessment

- [ ] Create arrays with `array`, `arange`, `linspace`, `zeros`, `ones`
- [ ] Inspect `shape`, `dtype`, `ndim`, `size`
- [ ] Index a 2-D array by row, column, and sub-matrix
- [ ] Filter with a boolean mask combining multiple conditions
- [ ] Compute a vectorised formula across an entire array (no loops)
- [ ] Use broadcasting to combine arrays of different shapes
- [ ] Aggregate along `axis=0` vs `axis=1`
- [ ] Generate reproducible random data with `default_rng`

## 🚀 Next step

Continue with **Notebook 8 — Matplotlib Basics**. Once you can compute the right numbers, the next job is communicating them — and a good chart is how you turn a NumPy array into a decision.